In [1]:
import numpy as np
import tensorflow as tf
from keras.datasets import cifar10
from keras.utils import np_utils
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D
from art.attacks.evasion import SaliencyMapMethod
from art.estimators.classification  import KerasClassifier
from sklearn.metrics import accuracy_score

In [2]:
tf.compat.v1.disable_eager_execution()

In [3]:
import tensorflow as tf
import json
# download mnist data and split into train and test sets
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()
# reshape data to fit model
X_train, X_test = X_train/255, X_test/255
# one-hot encode target column
y_train = tf.keras.utils.to_categorical(y_train)
y_test = tf.keras.utils.to_categorical(y_test)
# create model
model = Sequential()
model.add(Conv2D(32, (3, 3), padding='same', input_shape=(32, 32, 3), activation='relu'))
model.add(Conv2D(32, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Conv2D(64, (3, 3), padding='same', activation='relu'))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Flatten())
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(10, activation='softmax'))

In [4]:
# compile model using accuracy as a measure of model performance
model.compile(optimizer='adam', loss='categorical_crossentropy',
              metrics=['accuracy'])

# train model
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=5)

json.dump({'model': model.to_json()}, open("model.json", "w"))
model.save_weights("model_weights.h5")


Train on 50000 samples, validate on 10000 samples
Epoch 1/5
50000/50000 [==============================] - ETA: 0s - loss: 1.5183 - accuracy: 0.4430

C:\Users\nidhi.jain1\Anaconda3\lib\site-packages\keras\engine\training_v1.py:2333: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates = self.state_updates


50000/50000 [==============================] - 379s 8ms/sample - loss: 1.5183 - accuracy: 0.4430 - val_loss: 1.1863 - val_accuracy: 0.5838
Epoch 2/5
50000/50000 [==============================] - 527s 11ms/sample - loss: 1.1325 - accuracy: 0.5978 - val_loss: 0.9756 - val_accuracy: 0.6554
Epoch 3/5
50000/50000 [==============================] - 492s 10ms/sample - loss: 0.9917 - accuracy: 0.6519 - val_loss: 0.8799 - val_accuracy: 0.6920
Epoch 4/5
50000/50000 [==============================] - 398s 8ms/sample - loss: 0.8962 - accuracy: 0.6862 - val_loss: 0.8179 - val_accuracy: 0.7126
Epoch 5/5
50000/50000 [==============================] - 227s 5ms/sample - loss: 0.8326 - accuracy: 0.7087 - val_loss: 0.7617 - val_accuracy: 0.7361


In [5]:
#Create a KerasClassifier for the model
classifier = KerasClassifier(model=model, clip_values=(0, 1),  use_logits=False)
# Generate adversarial examples
x_test = X_test # your test data
y = y_test # the true labels for the test data
jsma = SaliencyMapMethod(classifier=classifier, theta=0.7, gamma=0.1)
#x_test_adv = jsma.generate(x=x_test, y=y)
x_test_adv = jsma.generate(x_test)

C:\Users\nidhi.jain1\Anaconda3\lib\site-packages\keras\engine\training_v1.py:2357: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,


JSMA:   0%|          | 0/10000 [00:00<?, ?it/s]

In [6]:
# Evaluate the model's accuracy on the test dataset
model.fit(X_train, y_train, batch_size=32, epochs=5, validation_data=(x_test_adv, y_test))
score1 = model.evaluate(x_test_adv, y_test, verbose=0)
print('Test accuracy:', score1[1])

x_test_adv
x_test
print(f"X_train shape: {x_test_adv.shape}")
print(f"y_train shape: {x_test.shape}")

Train on 50000 samples, validate on 10000 samples
Epoch 1/5
50000/50000 [==============================] - 307s 6ms/sample - loss: 0.7881 - accuracy: 0.7241 - val_loss: 1.7592 - val_accuracy: 0.4157
Epoch 2/5
50000/50000 [==============================] - 236s 5ms/sample - loss: 0.7470 - accuracy: 0.7384 - val_loss: 1.7863 - val_accuracy: 0.4111
Epoch 3/5
50000/50000 [==============================] - 157s 3ms/sample - loss: 0.7093 - accuracy: 0.7503 - val_loss: 1.6393 - val_accuracy: 0.4472
Epoch 4/5
50000/50000 [==============================] - 127s 3ms/sample - loss: 0.6857 - accuracy: 0.7596 - val_loss: 1.6541 - val_accuracy: 0.4406
Epoch 5/5
50000/50000 [==============================] - 97s 2ms/sample - loss: 0.6667 - accuracy: 0.7646 - val_loss: 1.6961 - val_accuracy: 0.4449
Test accuracy: 0.4449
X_train shape: (10000, 32, 32, 3)
y_train shape: (10000, 32, 32, 3)


In [7]:
# Evaluate the model's accuracy before adv attack on the test dataset
score1 = model.evaluate(x_test, y_test, verbose=0)
print('Test accuracy:', score1[1])

Test accuracy: 0.7682
